# Pruebas del Pipeline ETL (Lab 1B)
Este notebook te permite ejecutar y probar paso a paso el pipeline ETL, inspeccionando los DataFrames **completos** en cada etapa.

In [ ]:
import os
import sys
import pandas as pd
import sqlite3
from IPython.display import display

# Configurar pandas para mostrar todas las filas y columnas sin truncar
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Añadir src al path para importar nuestros módulos
sys.path.append(os.path.join(os.getcwd(), 'src'))

from extract import extract_all_transactions, extract_reference_data
from transform import profile_data, clean_and_harmonize, transform_and_integrate
from load import validate_data, load_data

### 1. Extracción (Extract)

In [ ]:
raw_dir = os.path.join(os.getcwd(), 'raw')

# Extraer transacciones
sales_df = extract_all_transactions(raw_dir)
print(f"Transacciones extraidas: {len(sales_df)}")
display(sales_df)

# Extraer catálogos
products_df, stores_df, promotions_df, targets_df = extract_reference_data(raw_dir)

print("\n--- Products ---")
display(products_df)
print("\n--- Stores ---")
display(stores_df)
print("\n--- Promotions ---")
display(promotions_df)
print("\n--- Targets ---")
display(targets_df)

### 2. Perfilamiento (Profile)

In [ ]:
profile_data(sales_df)

### 3. Limpieza y Homologación (Clean & Harmonize)

In [ ]:
clean_sales_df = clean_and_harmonize(sales_df)
print(f"Filas post-limpieza: {len(clean_sales_df)} (Se removieron {len(sales_df) - len(clean_sales_df)} filas invalidas)")
display(clean_sales_df)

### 4. Transformación e Integración (Transform & Integrate)

In [ ]:
integrated_df = transform_and_integrate(clean_sales_df, products_df, stores_df, promotions_df)
print(f"Columnas del dataset final: {len(integrated_df.columns)}")
display(integrated_df)

### 5. Validación (Validate)

In [ ]:
validate_data(integrated_df)
# Si todo está bien, imprimirá 'Validation passed successfully.'

### 6. Carga de Datos (Load)

In [ ]:
base_dir = os.getcwd()
load_data(integrated_df, targets_df, base_dir)

### 7. Verificación Final en Base de Datos

In [ ]:
db_path = os.path.join(base_dir, 'database', 'retail_analytics.db')
conn = sqlite3.connect(db_path)

# Ejecutamos un query rápido a la tabla final para ver TODOS los datos
query = "SELECT * FROM sales_analytics"
test_df = pd.read_sql(query, conn)
conn.close()

display(test_df)